##### Configuration and imports

In [ ]:
import ast
import uuid
import json
from delta.tables import DeltaTable
import re
import logging
import pyspark.sql.functions as F # import when, col, lit, to_date, to_timestamp, trim
from pyspark.sql import DataFrame
from datetime import date
import sys
import logging

In [ ]:
logger = logging.getLogger("TransactionalShared")

##### setup_logging() function

In [ ]:
# Define a function for logging
def setup_logging():
    '''
    Set up logging configuration for Fabric notebooks.
    '''
    # Create formatter
    FORMAT = "%(asctime)s UTC - %(levelname)s - %(message)s (%(name)s)"
    formatter = logging.Formatter(fmt=FORMAT)

    # Get root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.WARNING)

    # Check if handlers exist, if not add console handler
    if not root_logger.handlers:
        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setFormatter(formatter)
        console_handler.setLevel(logging.INFO)
        root_logger.addHandler(console_handler)
    else:
        # Apply formatter to existing handlers
        for handler in root_logger.handlers:
            handler.setFormatter(formatter)

##### Data transformation helpers

In [ ]:
# Define functions for data transformation

#1. Convert a string(column Names) to snake_case. Purpose: Standardize column naming > framework source: nb_load_to_silver
def to_snake_case(string):
    string = re.sub(r'([a-z])([A-Z0-9])', r'\1_\2', string)
    string = re.sub(r'([A-Z])([A-Z][a-z])', r'\1_\2', string)
    string = string.lower().strip()
    return string

#2. Remove leading/trailing spaces from all string column values. input: Spark DataFrame > nb_load_to_silver: nb_load_to_silver
def trim_all_string_columns(df):
    string_columns = [field.name for field in df.schema.fields if field.dataType.typeName() == 'string']
    df = df.select(
        *[F.trim(F.col(c)).alias(c) if c in string_columns else F.col(c) for c in df.columns]
    )
    return df

#3. Replace old dates: If date ≤ 1900-01-01 → replace with 1900-01-01. Input:DataFrame > nb_load_to_silver: nb_load_to_silver
def replace_old_dates(df):
    date_columns = [field.name for field in df.schema.fields if field.dataType.typeName() in('date','timestamp')]
    for date_column in date_columns:
        df = df.withColumn(
            date_column, 
		    F.when(
			    F.col(date_column) <= F.to_date(F.lit('1900-01-01')), F.to_date(F.lit('1900-01-01'), 'yyyy-MM-dd')
            )
			.otherwise(
                F.col(date_column)
            )
        )
    return df

#4. Define parent function that applies all cleaning steps to the DataFrame. > nb_load_to_silver: nb_load_to_silver
def clean_bronze_table(df, limit_rows_for_debugging=False):
    if limit_rows_for_debugging:
        df = df.limit(1000)
    snake_case_column_names = [to_snake_case(col) for col in df.columns]
    df = df.toDF(*snake_case_column_names)
    df = replace_old_dates(df)
    df = trim_all_string_columns(df)
    return df

#5. Add metadata columns to dataframe > nb_load_to_silver: nb_load_to_delta_table
def add_metadata(df: DataFrame, ingest_date: str, file_path: str, schema_name: str, table_name: str, lineage_id: str) -> DataFrame:
    return (
        df
        .withColumn("_ingest_time", F.current_timestamp()) # cast to hh:mm:00
        .withColumn("_ingest_date", F.lit(ingest_date))
        .withColumn("_source_file", F.lit(file_path)) #source_table
        .withColumn("_source_system", F.lit(schema_name))
        .withColumn("_table", F.lit(table_name)) #target_table
        .withColumn("_lineage_id", F.lit(lineage_id))
    )

#6. Removes duplicates from dataframe > new function
def remove_exact_duplicates(df):
    return df.dropDuplicates(df.columns)

##### Data load helpers

In [ ]:
#1. Check whether a business key has duplicates and display samples. > new function
def validate_duplicates(df, key_col="_hashed_pk", limit=10):
    dup_keys_df = (
        df.groupBy(key_col)
          .agg(F.count("*").alias("duplicate_count"))
          .filter(F.col("duplicate_count") > 1)
          .orderBy(F.col("duplicate_count").desc(), F.col(key_col))
    )

    total_dup_groups = dup_keys_df.count()
    print(f"Total duplicate {key_col} groups: {total_dup_groups}")
    display(dup_keys_df.limit(limit))

    if total_dup_groups > 0:
        dup_records_df = (
            df.join(dup_keys_df.select(key_col), on=key_col, how="inner")
              .orderBy(key_col)
        )
        display(dup_records_df.limit(limit))

#2. Framework-aligned target load helper. >  nb_load_to_silver: nb_load_to_silver
#   merge-scd1 updated only when the business row hash changes
def load_to_target(source_df_cleaned, target_table_path, target_load_strategy):

    # If target path does not exist, create it
    if not DeltaTable.isDeltaTable(spark, target_table_path):
        logger.info(f'Writing new table to {target_table_path}')
        source_df_cleaned.write.format("delta").mode("overwrite").save(target_table_path)

    else:
        match target_load_strategy:

            # Insert new _hashed_pk rows and update changed _hashed_row rows
            case "merge-scd1":
                logger.info(f'Running merge-scd1 into {target_table_path}')
                delta_table = DeltaTable.forPath(spark, target_table_path)

                (
                    delta_table.alias("t")
                    .merge(
                        source_df_cleaned.alias("s"),
                        "t._hashed_pk = s._hashed_pk"
                    )
                    .whenMatchedUpdateAll(
                        condition="NOT (t._hashed_row <=> s._hashed_row)"
                    )
                    .whenNotMatchedInsertAll()
                    .execute()
                )

            # Replace all target data
            case "overwrite":
                logger.info(f'Overwriting records in {target_table_path}')
                (
                    source_df_cleaned
                    .write.mode("overwrite")
                    .format("delta")
                    .option("overwriteSchema", "true")
                    .save(target_table_path)
                )

            # Add records to target data
            case "append":
                logger.info(f'Appending new records to {target_table_path}')
                (
                    source_df_cleaned
                    .write.mode("append")
                    .format("delta")
                    .option("mergeSchema", "true")
                    .save(target_table_path)
                )

            case _:
                raise ValueError(f"Unsupported target_load_strategy: {target_load_strategy}")

    # Return metrics from the latest Delta operation
    delta_table = DeltaTable.forPath(spark, target_table_path)
    metrics = delta_table.history(1).select("operationMetrics").first()["operationMetrics"]

    return {
        "rows_inserted": int(metrics.get("numTargetRowsInserted", metrics.get("numOutputRows", 0))),
        "rows_updated": int(metrics.get("numTargetRowsUpdated", 0)),
        "rows_read": int(metrics.get("numSourceRows", 0)),
        "rows_copied": int(metrics.get("numTargetRowsCopied", 0))
    }